# 03 — BCRD Family Basket | Gold Cost-of-Living KPIs

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Calculate cost-of-living KPIs **COL-001 to COL-003** from the BCRD family-basket cost series, at macro level (national, income quintile and region). Product-level prices remain the scope of the Pro Consumidor KPIs (CON-001 to CON-010).

## Lineage
`silver.bcrd_family_basket_monthly` → `gold.cost_of_living_metrics` (grain: **Month × segment**)

## KPIs
| KPI | Column | Formula |
|---|---|---|
| COL-001 Family Basket Cost | `basket_cost_dop` | Published BCRD value, DOP per month |
| COL-002 Basket Cost YoY % | `yoy_change_pct` | (Cost_t / Cost_t-12 − 1) × 100 |
| COL-003 Quintile 5 vs Quintile 1 Gap | `q5_q1_gap_dop`, `q5_q1_ratio` | Q5 − Q1 and Q5 / Q1, same month |

Also provided: `mom_change_pct` (supporting). COL-003 is a month-level value, so it is filled only on the `national` row of each month.

## Methodology flags
- Quintile and national values before Oct-2020 are a BCRD retropolation (`is_retropolated`). YoY values that use a retropolated month are flagged `uses_retropolated_value`.
- Windows without enough history are NULL with an `is_full_window_*` flag (same rule as D4).

**Status:** Development — not production-ready.

In [0]:
import uuid, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
SOURCE_TABLE = f"{CATALOG}.silver.bcrd_family_basket_monthly"
TARGET_TABLE = f"{CATALOG}.gold.cost_of_living_metrics"
RULE_RESULTS_TABLE = f"{CATALOG}.quality.dq_rule_results"
RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)

In [0]:
# ============================================================
# Step 1 — COL-001 and COL-002 per segment
# ============================================================
basket = spark.table(SOURCE_TABLE).withColumn("cost", F.col("basket_cost_dop").cast("double"))
w = Window.partitionBy("segment_type", "segment").orderBy("reference_month")

df = (
    basket
    .withColumn("_prev", F.lag("cost", 1).over(w))
    .withColumn("_prev_m", F.lag("reference_month", 1).over(w))
    .withColumn("_yago", F.lag("cost", 12).over(w))
    .withColumn("_yago_m", F.lag("reference_month", 12).over(w))
    .withColumn("_yago_retro", F.lag("is_retropolated", 12).over(w))
    .withColumn("is_full_window_mom", F.coalesce(F.months_between("reference_month", "_prev_m") == 1, F.lit(False)))
    .withColumn("is_full_window_yoy", F.coalesce(F.months_between("reference_month", "_yago_m") == 12, F.lit(False)))
    .withColumn("mom_change_pct", F.when(F.col("is_full_window_mom"), (F.col("cost") / F.col("_prev") - 1) * 100))
    .withColumn("yoy_change_pct", F.when(F.col("is_full_window_yoy"), (F.col("cost") / F.col("_yago") - 1) * 100))
    .withColumn("uses_retropolated_value", F.col("is_retropolated") | F.coalesce(F.col("_yago_retro"), F.lit(False)))
)

# ============================================================
# Step 2 — COL-003: Q5 vs Q1 gap (month level, on the national row)
# ============================================================
gap = (
    basket.filter(F.col("segment").isin("Quintil 1", "Quintil 5"))
    .groupBy("reference_month")
    .agg(F.max(F.when(F.col("segment") == "Quintil 5", F.col("cost"))).alias("_q5"),
         F.max(F.when(F.col("segment") == "Quintil 1", F.col("cost"))).alias("_q1"))
    .withColumn("q5_q1_gap_dop", F.col("_q5") - F.col("_q1"))
    .withColumn("q5_q1_ratio", F.col("_q5") / F.col("_q1"))
    .withColumn("segment_type", F.lit("national"))
    .select("reference_month", "segment_type", "q5_q1_gap_dop", "q5_q1_ratio")
)

out = (
    df.join(gap, ["reference_month", "segment_type"], "left")
    .select(
        "reference_month", "segment_type", "segment",
        F.col("cost").cast("decimal(14,2)").alias("basket_cost_dop"),
        F.round("mom_change_pct", 4).cast("decimal(10,4)").alias("mom_change_pct"), "is_full_window_mom",
        F.round("yoy_change_pct", 4).cast("decimal(10,4)").alias("yoy_change_pct"), "is_full_window_yoy",
        F.col("q5_q1_gap_dop").cast("decimal(14,2)").alias("q5_q1_gap_dop"),
        F.round("q5_q1_ratio", 4).cast("decimal(8,4)").alias("q5_q1_ratio"),
        "is_retropolated", "uses_retropolated_value",
        F.year("reference_month").alias("year"), F.month("reference_month").alias("month"),
        F.lit(RUN_ID).alias("_gold_run_id"), F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts"),
    )
)

In [0]:
# ============================================================
# Step 3 — Validation, write, documentation
# ============================================================
checks = []
def add(name, rtype, tested, failed):
    checks.append((RUN_ID, TARGET_TABLE, name, rtype, int(tested), int(failed),
                   "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))

n, src = out.count(), basket.count()
add("row_count_matches_silver", "enforced", n, abs(n - src))
add("unique_month_segment", "enforced", n, n - out.select("reference_month", "segment_type", "segment").distinct().count())
add("yoy_null_iff_incomplete", "enforced", n, out.filter(F.col("yoy_change_pct").isNull() == F.col("is_full_window_yoy")).count())
national = out.filter("segment_type = 'national'")
add("gap_present_on_national_rows", "enforced", national.count(), national.filter(F.col("q5_q1_gap_dop").isNull()).count())
add("q5_above_q1", "enforced", national.count(), national.filter(F.col("q5_q1_ratio") <= 1).count())

for c in checks:
    print(f"  {c[6]:<4} {c[2]:<32} tested={c[4]:,} failed={c[5]:,}")
assert not [c for c in checks if c[6] == "FAIL"], "Enforced checks failed; nothing written"

out.orderBy("segment_type", "segment", "reference_month").write.format("delta").mode("overwrite") \
   .option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)

spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS 'Gold: cost-of-living KPIs COL-001 to COL-003 from the BCRD family basket cost. Grain: month x segment (national, income quintile, region).'")
for col, comment in {
    "basket_cost_dop": "COL-001: BCRD family basket cost, DOP per month",
    "yoy_change_pct": "COL-002: % change vs same month previous year; NULL until 12 months of history",
    "q5_q1_gap_dop": "COL-003: Quintile 5 minus Quintile 1 basket cost, DOP (national rows only)",
    "q5_q1_ratio": "COL-003: Quintile 5 / Quintile 1 basket cost (national rows only)",
    "uses_retropolated_value": "True when the value or its year-ago comparison is a BCRD retropolation (before Oct-2020)",
}.items():
    spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN {col} COMMENT '{comment}'")

spark.createDataFrame(checks,
    "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp",
).write.mode("append").saveAsTable(RULE_RESULTS_TABLE)

In [0]:
last = spark.table(TARGET_TABLE).agg(F.max("reference_month")).first()[0]
snap = (spark.table(TARGET_TABLE).filter(F.col("reference_month") == last)
        .select("segment_type", "segment", "basket_cost_dop", "yoy_change_pct", "q5_q1_gap_dop", "q5_q1_ratio")
        .orderBy("segment_type", "segment"))
display(snap)

dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS", "rows": n, "latest_month": str(last),
    "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks},
    "snapshot": [{k: str(v) for k, v in r.asDict().items()} for r in snap.collect()],
}, ensure_ascii=False))